# Yeu cau 1: Data Pipeline
Lam sach, tien xu ly, trich xuat TF-IDF, luu dataset.

In [1]:
import pandas as pd
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

# Doc du lieu
df = pd.read_csv('spam.csv', encoding='latin-1', usecols=[0, 1])
df.columns = ['label', 'text']
print('Shape ban dau:', df.shape)
print(df['label'].value_counts())

Shape ban dau: (5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


In [2]:
# Lam sach du lieu
df.dropna(subset=['text', 'label'], inplace=True)
df.drop_duplicates(inplace=True)
df['label'] = df['label'].str.strip().str.lower()
print('Shape sau khi lam sach:', df.shape)

# Tien xu ly van ban
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
print(df[['text', 'clean_text', 'label', 'label_num']].head(3))

Shape sau khi lam sach: (5169, 2)
                                                text  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   

                                          clean_text label  label_num  
0  go until jurong point crazy available only in ...   ham          0  
1                            ok lar joking wif u oni   ham          0  
2  free entry in 2 a wkly comp to win fa cup fina...  spam          1  


In [3]:
# Feature Engineering: TF-IDF
tfidf = TfidfVectorizer(max_features=3000, stop_words='english')
X = tfidf.fit_transform(df['clean_text'])
y = df['label_num'].values

print('TF-IDF shape:', X.shape)

# Luu dataset da xu ly
df.to_csv('processed_data.csv', index=False)
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
import scipy.sparse as sp
sp.save_npz('X_tfidf.npz', X)
import numpy as np
np.save('y_labels.npy', y)

print('Da luu: processed_data.csv, tfidf_vectorizer.pkl, X_tfidf.npz, y_labels.npy')

TF-IDF shape: (5169, 3000)
Da luu: processed_data.csv, tfidf_vectorizer.pkl, X_tfidf.npz, y_labels.npy
